# NB01 — B1: coherencia interna de Attractor Convergence

**Parser L2 determinista** sobre la fuente canónica `main.tex` (no el PDF).
Módulo: `l2_parser.py` + `test_l2_parser.py` (7 tests, corren en el repo con
`python3 -m pytest analysis/test_l2_parser.py`).

 chequeado: 17 entornos LaTeX · Thm 1/2 con hipótesis→conclusiones ·
refs/cites · cifras clave vs `paper_data.vsm` · secciones del addendum.

## ① Inventario de estructura (17 entornos)

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))          # l2_parser.py vive en esta carpeta
from l2_parser import analyse

MAIN = Path("../main.tex")                    # local; en LXC114 → data/main.tex
if not MAIN.exists():
    MAIN = Path("data/main.tex")
r = analyse(MAIN)
print(f"source: {MAIN}  records: {r['env_count']}")
for rec in r["records"]:
    n_conc = len(rec["conclusions"])
    print(f"  L{rec['line']:5d}  {rec['kind']:<11} {(rec['name'] or ''):<55} "
          f"conc={n_conc} refs={len(rec['refs'])} label={rec['label']}")

source: ../main.tex  records: 17
  L  528  definition  Transition kernel                                       conc=0 refs=1 label=def:kernel
  L  590  remark      One object, three lenses                                conc=0 refs=3 label=None
  L  610  definition  Feasible region                                         conc=0 refs=0 label=def:omega
  L  654  definition  Contractive core                                        conc=0 refs=1 label=def:omega_a
  L  729  theorem     Hybrid invariance                                       conc=3 refs=3 label=thm:hybrid_invariance
  L  767  remark      The theorem is measured, not merely postulated          conc=0 refs=2 label=None
  L  778  remark      Two contraction numbers, two operators                  conc=0 refs=2 label=None
  L  806  definition  Lyapunov function                                       conc=0 refs=0 label=def:lyapunov
  L  824  theorem     Convergence in expectation                              conc=0 refs=1 label=th

## ② Thm 1 y Thm 2 — registros profundos

Thm 1 = statement (hipótesis) + **3 conclusiones numeradas** (Existence /
Convergence / Confinement). Thm 2 = statement + ecuación `eq:convergence`
(sin enumerate → conclusions=0 es correcto).

In [1]:
from l2_parser import parse_envs
recs = {x["label"]: x for x in parse_envs(MAIN.read_text(errors="replace")) if x["label"]}
t1 = recs["thm:hybrid_invariance"]
print(f"Thm 1 [{t1['name']}] L{t1['line']}  hyp_markers={t1['hyp_markers']}")
print(f"  statement: {t1['statement'][:180]}...")
for i, c in enumerate(t1["conclusions"], 1):
    print(f"  conclusion {i}: {c['head']}")
    print(f"      → {c['text'][:110]}...")
print(f"  refs: {t1['refs']}")
print()
t2 = recs["thm:convergence"]
print(f"Thm 2 [{t2['name']}] L{t2['line']}  hyp_markers={t2['hyp_markers']}")
print(f"  statement: {t2['statement'][:200]}")
print(f"  refs: {t2['refs']}  labels: {t2['labels']}")

Thm 1 [Hybrid invariance] L729  hyp_markers=['with', 'Then']
  statement: Write the stochastic update as $\sv(t{+}1) = f_0(\sv(t)) + \sigma\,\eta(t)$, with $f_0$ eventually contracting on the compact core $\Omega_A = \{\sv \in \Omega : \langle \sv, s\ran...
  conclusion 1: Existence (Krylov--Bogolyubov + Banach).
      → \textbf{Existence (Krylov--Bogolyubov + Banach).} The state space is compact and the update kernel is Feller (...
  conclusion 2: Convergence in expectation.
      → \textbf{Convergence in expectation.} The Lyapunov drift condition $\E[L(\sv(t{+}1)) - L(\sv(t))] \le 0$ (Theor...
  conclusion 3: Confinement (empirical).
      → \textbf{Confinement (empirical).} On $\Omega \setminus \Omega_A$ we do not claim per-step non-expansiveness --...
  refs: ['sec:sustained_noise', 'sec:thermal_cycling', 'thm:convergence']

Thm 2 [Convergence in expectation] L824  hyp_markers=['Under', 'if']
  statement: Under the axiom-governed dynamics~\eqref{eq:dynamics}, if $\alpha, \beta, \ga

## ③ Chequeos de coherencia

In [1]:
from l2_parser import dangling_refs, figure_tokens, addendum_sections
ADD = Path("../../../correspondence/fierro/addendum_omega/addendum.tex")
if not ADD.exists():
    ADD = Path("data/addendum.tex")
tex = MAIN.read_text(errors="replace")
print("dangling refs:", dangling_refs(tex) or "NONE \u2713")
tok = figure_tokens(tex)
print("figure tokens:", {k: ("\u2713" if v else "\u2717") for k, v in tok.items()})
secs = addendum_sections(ADD.read_text(errors="replace"))
print("addendum sections:", secs)

dangling refs: NONE ✓
figure tokens: {'lambda_max': '✓', 'thermal_ceiling': '✓', 'ns_ticks': '✓', 'psi_threshold': '✓', 'soplo_freq': '✓', 'basin_floor': '✓'}
addendum sections: ['Context', '1. Convexity of $\\Omega$', '2. Continuity and Invariance of $f_0$', '3. Stochastic Perturbations $\\sigma \\cdot \\eta(t)$', 'Integration with the Manuscript']


## ④ Hallazgos B1

| ID | Hallazgo | Severidad | Estado |
|---|---|---|---|
| **D1** | `\date` del paper: *"revised June 2026"* pero el material incluye las enmiendas de **julio** (footnote Fierro, Thm 1 sin convexidad, FIERRO-L-BOUND) — README F6 ya lo llamó "version skew" | baja (editorial) | **abierto** — resuelto con el freeze v0.9 pendiente del paquete de envío |
| — | dangling refs | — | **NONE ✓** |
| — | cifras clave (λmax, 0.9377, N=3000, θPSI, soplo, basin) vs paper_data | — | **6/6 ✓** |
| — | Thm 1 descargas Q1/Q2/Q3 (convexity-free · no-claim non-expansive · confinement) | — | **✓ embebidas en conclusiones** |

**B1 cerrado** — la coherencia interna del paper está limpia salvo D1 (editorial).


## ⑤ Próximos pasos

- **B2**: inventario de A0/A0.1 en el paper → clasificar como *interface de la
  capa agente* (fuera de certificación — decisión S5 2026-10-09).
- **C (NB02)**: correlación Qi ↔ addendum §i ↔ cláusulas Thm 1/2 (layer2b).
- **D (NB03)**: reporte del conjunto + decisión de envío (freeze v0.9).
